In [37]:
import fredapi
from dotenv import load_dotenv
import os
import pandas as pd
import sqlalchemy as db
from sqlalchemy import text

In [38]:
load_dotenv()
api_key = os.getenv("API_KEY")

In [39]:
fred = fredapi.Fred(api_key)

In [40]:
data = fred.get_series('IRLTLT01USM156N')

In [41]:
data_vintage = fred.get_series_vintage_dates('IRLTLT01USM156N')

In [42]:
pd.to_datetime(data_vintage)

DatetimeIndex(['2013-06-03', '2013-07-01', '2013-08-01', '2013-08-21',
               '2013-10-01', '2013-11-01', '2013-12-02', '2014-02-03',
               '2014-11-03', '2014-12-01',
               ...
               '2025-05-15', '2025-06-16', '2025-07-15', '2025-08-15',
               '2025-09-15', '2025-10-15', '2025-11-17', '2025-12-15',
               '2026-01-15', '2026-02-16'],
              dtype='datetime64[us]', length=135, freq=None)

In [43]:
data = data.dropna()

In [44]:
data.head()

1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
dtype: float64

In [45]:
data.tail()

2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
dtype: float64

In [46]:
data.describe

<bound method NDFrame.describe of 1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
              ... 
2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
Length: 874, dtype: float64>

In [47]:
type(data)

pandas.Series

In [48]:
print(fred.get_series_all_releases('IRLTLT01USM156N'))

           realtime_start                 date value
0     2024-04-10 00:00:00  1953-04-01 00:00:00  2.83
1     2024-04-10 00:00:00  1953-05-01 00:00:00  3.05
2     2024-04-10 00:00:00  1953-06-01 00:00:00  3.11
3     2024-04-10 00:00:00  1953-07-01 00:00:00  2.93
4     2024-04-10 00:00:00  1953-08-01 00:00:00  2.95
...                   ...                  ...   ...
2070  2025-10-15 00:00:00  2025-09-01 00:00:00  4.12
2071  2025-11-17 00:00:00  2025-10-01 00:00:00  4.06
2072  2025-12-15 00:00:00  2025-11-01 00:00:00  4.09
2073  2026-01-15 00:00:00  2025-12-01 00:00:00  4.14
2074  2026-02-16 00:00:00  2026-01-01 00:00:00  4.21

[2075 rows x 3 columns]


In [49]:
# Approximate fixed lags in days from value date to publication as FREDAPI doesn't offers real publish dates for old publications
rate_lags = {
    'UNRATE': 45,  # ~6 weeks after reference month end
    'CPIAUCSL': 15,  # ~2 weeks after reference month end
    'M2SL': 30,
    'WALCL': 7,
    'NFCI': 7,
    'ICSA': 5,  # Thursdays, covers prior week
}

def get_fred_data(serie):
    '''
    Process and store in the db the indicator
    '''
    #get the data
    data = fred.get_series(serie)
    #drop nan values
    data = data.dropna()

    if serie in rate_lags:
        data.index = data.index + pd.DateOffset(days=rate_lags[serie])

    # convert Series to DataFrame
    data = data.reset_index()
    data.columns = ['date', 'value']

    # remove timezone info
    data['date'] = pd.to_datetime(data['date']).dt.tz_localize(None)

    # add series name
    data['serie'] = serie

    return data

get_fred_data('M2SL')

,date,value,serie
0,1959-01-31,286.6,M2SL
1,1959-03-03,287.7,M2SL
2,1959-03-31,289.2,M2SL
3,1959-05-01,290.1,M2SL
4,1959-05-31,292.2,M2SL
...,...,...,...
799,2025-08-31,22108.2,M2SL
800,2025-10-01,22211.9,M2SL
801,2025-10-31,22297.8,M2SL
802,2025-12-01,22322.1,M2SL


In [50]:
engine = db.create_engine('sqlite:///../data/data.db')
with engine.connect() as conn:
    conn.execute(text('ALTER TABLE Macro DROP COLUMN vintage_date'))
    conn.commit()